# DeepSORVF — Phase 3: C0, C1, C2 (SMD NIR Dataset)

Benchmark d'adaptation de domaine : exécute les configurations **sans fusion AIS** sur le dataset
**Singapore Maritime Domain (SMD)** en infrarouge proche (NIR). Objectif : prouver la transférabilité
des modules de perception visuelle du pipeline DeepSORVF du RGB (FVessel) au NIR (SMD).

- **C0** : Baseline YOLOX (sans Kolomverse, sans Static Filter)
- **C1** : + Kolomverse (ensemble NMS)
- **C2** : + Static Filter (filtrage sea clutter)

Résultats sauvegardés dans `ablation_results/phase3/`.

**Sécurisé** : skip les clips/configs déjà traités. Relancer sans problème.

---
## 0. Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Extract zip and prepare project
import os, zipfile, shutil, glob

ZIP_PATH = '/content/drive/MyDrive/DeepSORVF_Colab.zip'
PROJECT_ROOT = '/content/DeepSORVF'

# Always re-extract to avoid stale code
if os.path.exists(PROJECT_ROOT):
    shutil.rmtree(PROJECT_ROOT)

print('Extracting zip...')
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall(PROJECT_ROOT)

# Clear __pycache__
for p in glob.glob(os.path.join(PROJECT_ROOT, '**', '__pycache__'), recursive=True):
    shutil.rmtree(p)

# --- Prepare SMD clips from Drive ---
SMD_DRIVE = '/content/drive/MyDrive/SMD_Data'
local_clips = os.path.join(PROJECT_ROOT, 'data', 'clips')
os.makedirs(local_clips, exist_ok=True)

# SMD videos to process (verified by verif_code.py)
SMD_VIDEOS = {
    'NIR': ['MVI_1463_NIR.avi', 'MVI_1468_NIR.avi', 'MVI_1520_NIR.avi', 'MVI_1521_NIR.avi'],
    'VIS_Onshore': ['MVI_1469_VIS.avi', 'MVI_1474_VIS.avi', 'MVI_1478_VIS.avi'],
    'VIS_Onboard': ['MVI_0790_VIS_OB.avi'],
}

# Dummy camera_para (SMD has no calibration data; AIS is disabled so unused)
DUMMY_CAMERA_PARA = '[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]'

clips_created = []
clips_skipped = []

for category, video_list in SMD_VIDEOS.items():
    src_video_dir = os.path.join(SMD_DRIVE, category, 'Videos')
    for video_name in video_list:
        # Clip name = video base name (e.g. MVI_1463_NIR)
        base_name = os.path.splitext(video_name)[0]
        clip_dir = os.path.join(local_clips, base_name)

        # Skip if already prepared
        if os.path.exists(os.path.join(clip_dir, 'camera_para.txt')):
            clips_skipped.append(base_name)
            continue

        os.makedirs(clip_dir, exist_ok=True)

        # Copy video with simple name (no timestamp needed — file_read fallback handles it)
        src_video = os.path.join(src_video_dir, video_name)
        dst_video = os.path.join(clip_dir, f'{base_name}.avi')
        if os.path.exists(src_video):
            shutil.copy2(src_video, dst_video)
        else:
            print(f'  WARNING: {src_video} not found, skipping {base_name}')
            shutil.rmtree(clip_dir)
            continue

        # Create dummy camera_para.txt
        with open(os.path.join(clip_dir, 'camera_para.txt'), 'w') as f:
            f.write(DUMMY_CAMERA_PARA)

        # Create empty ais/ directory (required by file_read.ais_initial)
        os.makedirs(os.path.join(clip_dir, 'ais'), exist_ok=True)

        clips_created.append(base_name)

print(f'SMD clips prepared: {len(clips_created)} created, {len(clips_skipped)} skipped')
if clips_created:
    print(f'  Created: {clips_created}')
if clips_skipped:
    print(f'  Skipped (already exist): {clips_skipped}')

In [ ]:
# Install dependencies
!pip install ultralytics==8.4.121 filterpy lap easydict geopy pyproj fastdtw loguru scikit-image

import torch
print(f'PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')

In [ ]:
# Verify clips and weights
clips_dir = os.path.join(PROJECT_ROOT, 'data', 'clips')
CLIPS = sorted([d for d in os.listdir(clips_dir)
                if os.path.isdir(os.path.join(clips_dir, d))])
print(f'Detected {len(CLIPS)} clips: {CLIPS}\n')

for clip in CLIPS:
    clip_path = os.path.join(clips_dir, clip)
    files = os.listdir(clip_path)
    video = [f for f in files if f.endswith(('.mp4', '.avi'))]
    ais_dir = os.path.join(clip_path, 'ais')
    ais_count = len(os.listdir(ais_dir)) if os.path.isdir(ais_dir) else 0
    has_camera = os.path.exists(os.path.join(clip_path, 'camera_para.txt'))
    print(f'  {clip}: video={video[0] if video else "MISSING"}, ais={ais_count}, camera_para={has_camera}')

weights_dir = os.path.join(PROJECT_ROOT, 'weights')
print()
for name in ['best.pt', 'YOLOX-final.pth', 'ckpt.t7']:
    path = os.path.join(weights_dir, name)
    if os.path.exists(path):
        print(f'  {name}: {os.path.getsize(path)/1e6:.1f} MB')
    else:
        print(f'  {name}: MISSING')

---
## 1. Quick Test
Vérifie le pipeline sur une vidéo SMD NIR avant l'ablation complète.

In [ ]:
# Quick test: C0 on first SMD clip (50 frames)
import sys
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

from run_ablation import run_pipeline

test_clip = CLIPS[0]
print(f'=== Quick test: C0 / {test_clip} (50 frames) ===')
stats = run_pipeline(
    clip_name=test_clip,
    result_dir=os.path.join(PROJECT_ROOT, 'quick_test'),
    config_name='C0',
    max_frames=50,
    use_ensemble=False, use_static_filter=False,
    ais_enabled=False, anti=0,
)
print(f'Result: {stats}')

---
## 2. Phase 3 — C0, C1, C2 (SMD NIR)
Skip les configs déjà traitées. Relancer sans problème.

In [ ]:
# Phase 3: C0, C1, C2 on SMD — skip existing, merge results
import json

PHASE3 = {
    'C0': dict(use_ensemble=False, use_static_filter=False, ais_enabled=False, anti=0),
    'C1': dict(use_ensemble=True,  use_static_filter=False, ais_enabled=False, anti=0),
    'C2': dict(use_ensemble=True,  use_static_filter=True,  ais_enabled=False, anti=0),
}

RESULT_DIR = os.path.join(PROJECT_ROOT, 'ablation_results', 'phase3')
os.makedirs(RESULT_DIR, exist_ok=True)

# Load existing results
summary_path = os.path.join(RESULT_DIR, 'phase3_summary.json')
phase3_results = []
if os.path.exists(summary_path):
    with open(summary_path) as f:
        phase3_results = json.load(f)
    print(f'Loaded {len(phase3_results)} existing results')

done = set((r['clip'], r['config']) for r in phase3_results if 'error' not in r)

for clip in CLIPS:
    for config_name, flags in PHASE3.items():
        if (clip, config_name) in done:
            print(f'  SKIP {config_name} / {clip} (already done)')
            continue
        res_dir = os.path.join(RESULT_DIR, clip, config_name)
        print(f'\n--- {config_name} / {clip} ---')
        try:
            stats = run_pipeline(
                clip_name=clip,
                result_dir=res_dir,
                config_name=config_name,
                **flags
            )
            phase3_results.append(stats)
        except Exception as e:
            print(f'  ERROR: {e}')
            phase3_results.append({'config': config_name, 'clip': clip, 'error': str(e)})

with open(summary_path, 'w') as f:
    json.dump(phase3_results, f, indent=2)

print(f'\n=== Phase 3 total: {len(phase3_results)} runs ===')

---
## 3. Résultats

In [ ]:
# Print summary table
import json

RESULT_DIR = os.path.join(PROJECT_ROOT, 'ablation_results', 'phase3')
summary_path = os.path.join(RESULT_DIR, 'phase3_summary.json')

if os.path.exists(summary_path):
    with open(summary_path) as f:
        results = json.load(f)

    print(f"{'Config':<6} {'Clip':<16} {'Frames':>8} {'Det-Sec':>8} {'Time':>8} {'ms/frm':>8}")
    print('-' * 64)
    for r in results:
        if 'error' in r:
            print(f"{r['config']:<6} {r['clip']:<16} ERROR: {r['error']}")
        else:
            print(f"{r['config']:<6} {r['clip']:<16} {r['total_frames']:>8} {r['detection_seconds']:>8} {r['wall_time_s']:>7.1f}s {r['avg_ms_per_frame']:>7.1f}")
    print(f'\nTotal: {len(results)} runs')
else:
    print('No results yet')

---
## 4. Assertions de validation
Vérifications requises par le protocole SMD (guide_developpement_smd.md).

In [ ]:
# Validation assertions
import json, csv, os

RESULT_DIR = os.path.join(PROJECT_ROOT, 'ablation_results', 'phase3')
summary_path = os.path.join(RESULT_DIR, 'phase3_summary.json')

if not os.path.exists(summary_path):
    print('No results to validate')
else:
    with open(summary_path) as f:
        results = json.load(f)

    print('=== Assertion 1: Ablation logs must be empty (AIS disabled) ===')
    for clip in CLIPS:
        for config_name in PHASE3:
            log_path = os.path.join(RESULT_DIR, clip, config_name,
                                   f'{clip}_{config_name}_ablation_log.csv')
            if os.path.exists(log_path):
                with open(log_path) as f:
                    lines = sum(1 for _ in f) - 1  # minus header
                if lines > 0:
                    print(f'  [WARNING] {config_name}/{clip}: {lines} lines in ablation log — AIS bridge may not be disabled!')
                else:
                    print(f'  [OK] {config_name}/{clip}: ablation log empty')
            else:
                print(f'  [OK] {config_name}/{clip}: no ablation log (AIS not used)')

    print('\n=== Assertion 2: Static Filter effect (C2 vs C1 detection count) ===')
    c1_counts = {}
    c2_counts = {}
    for r in results:
        if 'error' in r:
            continue
        det_path = os.path.join(RESULT_DIR, r['clip'], r['config'],
                                f"{r['clip']}_{r['config']}_detection.txt")
        if os.path.exists(det_path):
            count = sum(1 for _ in open(det_path))
            if r['config'] == 'C1':
                c1_counts[r['clip']] = count
            elif r['config'] == 'C2':
                c2_counts[r['clip']] = count

    for clip in CLIPS:
        if clip in c1_counts and clip in c2_counts:
            if c1_counts[clip] == c2_counts[clip]:
                print(f'  [WARNING] {clip}: C1={c1_counts[clip]} == C2={c2_counts[clip]} — Static Filter Too Permissive for NIR Domain')
            else:
                delta = c1_counts[clip] - c2_counts[clip]
                print(f'  [OK] {clip}: C1={c1_counts[clip]}, C2={c2_counts[clip]} (filtered {delta})')
        else:
            print(f'  [SKIP] {clip}: missing detection data for comparison')

---
## 5. Download Results

In [ ]:
# Zip and download phase3 results
import shutil
from google.colab import files

zip_path = '/tmp/phase3_results.zip'
shutil.make_archive('/tmp/phase3_results', 'zip',
                    os.path.join(PROJECT_ROOT, 'ablation_results', 'phase3'))
print(f'Zipped: {os.path.getsize(zip_path)/1e6:.1f} MB')
files.download(zip_path)